In [41]:
!pip -q install plotly ipywidgets

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from ipywidgets import interact, widgets
from IPython.display import clear_output, display

In [31]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [32]:
path = "/content/drive/MyDrive/Data Vis/owid-co2-data.csv"


In [33]:
df = pd.read_csv(path)
df.head()

,country,year,iso_code,population,gdp,cement_co2,cement_co2_per_capita,co2,co2_growth_abs,co2_growth_prct,...,share_global_other_co2,share_of_temperature_change_from_ghg,temperature_change_from_ch4,temperature_change_from_co2,temperature_change_from_ghg,temperature_change_from_n2o,total_ghg,total_ghg_excluding_lucf,trade_co2,trade_co2_share
0,Afghanistan,1750,AFG,2802560.0,NaN,0.0,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Afghanistan,1751,AFG,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Afghanistan,1752,AFG,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Afghanistan,1753,AFG,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Afghanistan,1754,AFG,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [34]:
#columns used for CO2 + Energy + GDP dashboard

cols = [
    "country", "iso_code", "year", "co2", "co2_per_capita", "gdp",
    "population", "primary_energy_consumption", "coal_co2",
    "oil_co2", "gas_co2"
]

df = df[cols].copy()

#filter years >=1990 (better coverage + easier trend discussion)
df = df[df["year"] >= 1990].copy()

#remove OWID aggregate regions
df = df[~df["iso_code"].astype(str).str.startswith("OWID_")].copy()

df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8883 entries, 240 to 50410
Data columns (total 11 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   country                     8883 non-null   object 
 1   iso_code                    7630 non-null   object 
 2   year                        8883 non-null   int64  
 3   co2                         8617 non-null   float64
 4   co2_per_capita              8068 non-null   float64
 5   gdp                         5421 non-null   float64
 6   population                  8015 non-null   float64
 7   primary_energy_consumption  7394 non-null   float64
 8   coal_co2                    5442 non-null   float64
 9   oil_co2                     8143 non-null   float64
 10  gas_co2                     4922 non-null   float64
dtypes: float64(8), int64(1), object(2)
memory usage: 832.8+ KB


In [35]:
#line chart: global CO2 trend
global_ts = df.groupby("year", as_index = False)["co2"].sum(min_count=1)

fig = px.line(
    global_ts, x = "year", y = "co2",
    title = "Global CO2 emissions over time (1990 onwards)"
)
fig.update_layout(yaxis_title="CO2 (million tonnes)")
fig.show()


In [36]:
#Chloropeth map : CO2 by country
map_df = df.dropna(subset=["iso_code", "co2"])

fig = px.choropleth(
    map_df,
    locations = "iso_code",
    color = "co2",
    hover_name = "country",
    animation_frame = "year",
    color_continuous_scale = "Reds",
    title = "CO2 emissions by country (1990 onwards)"
)

fig.update_layout(margin=dict(l = 0, r = 0, t = 60, b = 0))
fig.show()

In [37]:
#scatter: GDP vs CO2 (year slider)
scatter_df = df.dropna(subset=["gdp", "co2", "population"])

scatter_df = df.dropna(subset=["gdp", "co2", "population"])

fig = px.scatter(
    scatter_df,
    x = "gdp",
    y = "co2",
    size = "population",
    hover_name = "country",
    animation_frame = "year",
    animation_group = "country",
    log_x = True,
    title = "GDP vs CO2 emissions (animated by year)",
    labels = {"gdp" : "GDP", "co2" : "CO2 (million tonnes)"}
)

fig.show()

In [47]:
#stacked area: Fossil CO2 mix (coal / oil / gas) by country
mix_df = df.dropna(subset=["coal_co2", "oil_co2", "gas_co2"]).copy()

# Pick Top 30 countries by average total fossil CO2 (keeps it fast + clean)
mix_df["fossil_total"] = mix_df["coal_co2"] + mix_df["oil_co2"] + mix_df["gas_co2"]
top_countries = (
    mix_df.groupby("country")["fossil_total"]
    .mean()
    .sort_values(ascending=False)
    .head(30)
    .index
    .tolist()
)

mix_top = mix_df[mix_df["country"].isin(top_countries)].copy()
top_countries = sorted(top_countries)

fig = go.Figure()
trace_map = {}

for c in top_countries:
    d = mix_top[mix_top["country"] == c].sort_values("year")
    start = len(fig.data)

    fig.add_trace(go.Scatter(x=d["year"], y=d["coal_co2"], stackgroup="one", name="Coal CO₂", visible=False))
    fig.add_trace(go.Scatter(x=d["year"], y=d["oil_co2"],  stackgroup="one", name="Oil CO₂",  visible=False))
    fig.add_trace(go.Scatter(x=d["year"], y=d["gas_co2"],  stackgroup="one", name="Gas CO₂",  visible=False))

    trace_map[c] = [start, start+1, start+2]

# default country
default = "Malaysia" if "Malaysia" in top_countries else top_countries[0]
for i in trace_map[default]:
    fig.data[i].visible = True

buttons = []
for c in top_countries:
    visible = [False] * len(fig.data)
    for i in trace_map[c]:
        visible[i] = True
    buttons.append(dict(
        label=c,
        method="update",
        args=[{"visible": visible},
              {"title": f"Fossil source CO₂ mix over time — {c} (Top 30 emitters)"}]
    ))

fig.update_layout(
    title=f"Fossil source CO₂ mix over time — {default} (Top 30 emitters)",
    xaxis_title="Year",
    yaxis_title="CO₂ (million tonnes)",
    updatemenus=[dict(buttons=buttons, direction="down", x=1.02, y=1, xanchor="left", yanchor="top")],
    margin=dict(l=50, r=220, t=70, b=50)
)

fig.show()

In [48]:
#bubble: priary energy consumption vs CO2

bubble_df = df.dropna(subset=["primary_energy_consumption", "co2", "population"]).copy()

fig = px.scatter(
    bubble_df,
    x = "primary_energy_consumption",
    y = "co2",
    size = "population",
    hover_name = "country",
    animation_frame = "year",
    animation_group= "country",
    log_x = True,
    title = "Primary energy consumption vs CO2 emissions (animated by year)",
    labels = {
        "primary_energy_consumption" : "Primary energy consumption",
        "co2" : "CO2 (million tonnes)"
    }
)

fig.show()